# Cicero Digital – Session 4: LatinCy → CoNLL-U → BERTopic

In diesem Notebook gehst du **einen Schritt weiter** als im spaCy/LatinCy-Notebook:

1. Du lädst das in der vorherigen Session erzeugte Brief-Dataset.
2. Du analysierst die Briefe mit **LatinCy / spaCy**.
3. Du speicherst die linguistische Annotation als **CoNLL-U**.
4. Du bereitest die Texte für **Topic Modelling** vor.
5. Du modellierst Topics mit **BERTopic** und visualisierst sie auf verschiedene Arten.

Die Idee ist methodisch bewusst einfach gehalten: erst eine **transparente Vorverarbeitung**, dann ein **exploratives Topic Modelling**.


## 0. Setup

Wir brauchen:

- `pandas` für Tabellen
- `spaCy` + möglichst `LatinCy` für die lateinische Vorverarbeitung
- `bertopic` für Topic Modelling
- `sentence-transformers` für Embeddings
- `matplotlib` und `seaborn` für Visualisierungen

**Hinweis:** Gerade bei BERTopic kann die Installation je nach Umgebung etwas dauern.


In [ ]:
from pathlib import Path
import re
import math
import json
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import spacy
from spacy import displacy

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)


### Optional: Installation (falls nötig)

Diese Zelle ist nur für lokale Umgebungen gedacht. In vielen Jupyter-Setups ist bereits alles installiert.


In [ ]:
# OPTIONAL: nur ausführen, falls Pakete fehlen
# !pip install spacy pandas numpy matplotlib seaborn conllu bertopic sentence-transformers umap-learn hdbscan
# Optional LatinCy-Modell:
# !pip install https://huggingface.co/latincy/la_core_web_lg/resolve/main/la_core_web_lg-any-py3-none-any.whl


## 1. Brief-CSV einlesen

Wir gehen wie im vorherigen Notebook davon aus, dass pro Brief **eine Zeile** in einer CSV-Datei liegt, etwa:

`outputs/cicero_letters.csv`


In [ ]:
csv_path = Path("outputs") / "cicero_letters.csv"
df = pd.read_csv(csv_path)
df.head(3)


In [ ]:
df.columns.tolist()


Ein paar schnelle Checks:
- Wie viele Briefe haben wir?
- Welche Subkorpora gibt es?
- Gibt es leere Texte?


In [ ]:
len(df), df["corpus"].value_counts().head(20), (df["text"].fillna("").str.len() == 0).sum()


## 2. Arbeitsverzeichnisse anlegen

Wir speichern die Ergebnisse in einem eigenen Ordner.


In [ ]:
base_out = Path("outputs")
conllu_dir = base_out / "conllu_letters"
topic_dir = base_out / "topic_modeling"

conllu_dir.mkdir(parents=True, exist_ok=True)
topic_dir.mkdir(parents=True, exist_ok=True)

conllu_dir, topic_dir


## 3. LatinCy initialisieren

Wir versuchen zuerst, ein lateinisches Modell zu laden. Wenn das nicht klappt, fällt das Notebook auf eine minimale `blank("la")`-Pipeline zurück.

**Wichtig:** Für gute Lemmata, POS-Tags und vor allem Dependenzen brauchst du ein echtes lateinisches Modell.


In [ ]:
def load_nlp():
    model_candidates = [
        "la_core_web_lg",
        "la_core_web_md",
        "la_core_web_sm",
    ]
    for name in model_candidates:
        try:
            nlp = spacy.load(name)
            print(f"Loaded model: {name}")
            return nlp
        except Exception:
            pass

    print("Kein LatinCy-Modell gefunden. Nutze spaCy blank('la') als Fallback.")
    nlp = spacy.blank("la")
    if "sentencizer" not in nlp.pipe_names:
        nlp.add_pipe("sentencizer")
    return nlp

nlp = load_nlp()
nlp.pipe_names


## 4. Einen Brief probeweise analysieren

Wir schauen uns zuerst **einen** Brief an, bevor wir die ganze Sammlung durchlaufen.


In [ ]:
example = df.iloc[0]
example[["corpus", "book_n", "letter_n"] if {"corpus", "book_n", "letter_n"}.issubset(df.columns) else df.columns[:3]].to_dict()


In [ ]:
doc = nlp(str(example["text"]))
[(t.text, t.lemma_, t.pos_, t.dep_) for t in doc[:25]]


In [ ]:
list(doc.sents)[:2]


### Optional: Parsingbaum anzeigen

Diese Visualisierung funktioniert am besten, wenn das geladene Modell wirklich Dependenzen liefert.


In [ ]:
first_sent = list(doc.sents)[0]
displacy.render(first_sent, style="dep", jupyter=True)


## 5. CoNLL-U Export vorbereiten

Jetzt bauen wir eine kleine Funktion, die einen `spaCy`-`Doc` in ein **CoNLL-U-ähnliches Format** überführt.

Dabei gilt:
- `FORM` = Oberflächenform
- `LEMMA` = Lemma
- `UPOS` = Universal POS
- `HEAD` und `DEPREL` nur, falls vorhanden
- Metadaten (`sent_id`, `text`) werden als Kommentarzeilen vor jeden Satz geschrieben


In [ ]:
def escape_conllu_text(text: str) -> str:
    return text.replace("\n", " ").strip()

def token_to_conllu_line(token):
    # HEAD: CoNLL-U zählt ab 1, ROOT = 0
    if token.dep_ and token.dep_.upper() != "ROOT":
        head = token.head.i - token.sent.start + 1
    elif token.dep_.upper() == "ROOT":
        head = 0
    else:
        head = "_"

    fields = [
        str(token.i - token.sent.start + 1),                    # ID
        token.text if token.text else "_",                     # FORM
        token.lemma_ if token.lemma_ else "_",                 # LEMMA
        token.pos_ if token.pos_ else "_",                     # UPOS
        token.tag_ if token.tag_ else "_",                     # XPOS
        "_",                                                   # FEATS
        str(head) if head != "_" else "_",                     # HEAD
        token.dep_ if token.dep_ else "_",                     # DEPREL
        "_",                                                   # DEPS
        "_"                                                    # MISC
    ]
    return "\t".join(fields)

def doc_to_conllu(doc, doc_id="doc"):
    parts = []
    for sent_i, sent in enumerate(doc.sents, start=1):
        parts.append(f"# sent_id = {doc_id}_s{sent_i}")
        parts.append(f"# text = {escape_conllu_text(sent.text)}")
        for token in sent:
            if token.is_space:
                continue
            parts.append(token_to_conllu_line(token))
        parts.append("")
    return "\n".join(parts)


In [ ]:
print(doc_to_conllu(doc, doc_id="example")[:1200])


## 6. Gesamtkorpus mit LatinCy verarbeiten

Wir parsen jetzt alle Briefe. Das kann je nach Korpusgrösse und Modell etwas dauern.

Zusätzlich speichern wir:
- pro Brief eine `.conllu`-Datei
- eine Metadatentabelle mit ein paar Kennzahlen


In [ ]:
def make_doc_id(row, idx):
    corpus = str(row.get("corpus", "unknown")).strip()

    book_n = row.get("book_n")
    letter_n = row.get("letter_n")

    book_n = "x" if pd.isna(book_n) else str(book_n).strip()
    letter_n = str(idx) if pd.isna(letter_n) else str(letter_n).strip()

    return f"{corpus}_b{book_n}_l{letter_n}_r{idx}"

texts = df["text"].fillna("").astype(str).tolist()
doc_ids = [make_doc_id(row, idx) for idx, (_, row) in enumerate(df.iterrows())]

docs = []
meta_rows = []

for doc_id, (_, row), doc in zip(doc_ids, df.iterrows(), nlp.pipe(texts, batch_size=16)):
    docs.append(doc)

    conllu_text = doc_to_conllu(doc, doc_id=doc_id)
    (conllu_dir / f"{doc_id}.conllu").write_text(conllu_text, encoding="utf-8")

    tokens = [t for t in doc if not t.is_space]
    alpha_tokens = [t for t in tokens if t.is_alpha]
    sentences = list(doc.sents)

    meta_rows.append({
        "doc_id": doc_id,
        "corpus": row.get("corpus", None),
        "book_n": row.get("book_n", None),
        "letter_n": row.get("letter_n", None),
        "date_when": row.get("date_when", None),
        "n_tokens": len(tokens),
        "n_alpha_tokens": len(alpha_tokens),
        "n_sentences": len(sentences),
        "mean_sent_len": round(np.mean([len([t for t in s if not t.is_space]) for s in sentences]), 2) if sentences else np.nan
    })

meta = pd.DataFrame(meta_rows)
meta.head()


In [ ]:
meta_path = base_out / "cicero_letters_latincy_metadata.csv"
meta.to_csv(meta_path, index=False)

meta_path


### Optional: alle Briefe auch als **eine** grosse CoNLL-U-Datei speichern


In [ ]:
combined_conllu_path = base_out / "cicero_letters_all.conllu"

with combined_conllu_path.open("w", encoding="utf-8") as f:
    for doc_id, doc in zip(doc_ids, docs):
        f.write(doc_to_conllu(doc, doc_id=doc_id))
        f.write("\n")

combined_conllu_path


## 7. Kleine Qualitätskontrolle der Annotation

Bevor wir Topic Modelling machen, schauen wir uns an:
- POS-Verteilung insgesamt
- mittlere Satzlänge nach Subkorpus


In [ ]:
pos_counter = Counter()
for doc in docs:
    for tok in doc:
        if tok.is_space:
            continue
        pos_counter[tok.pos_ if tok.pos_ else "UNK"] += 1

pos_df = (
    pd.Series(pos_counter)
    .sort_values(ascending=False)
    .rename_axis("pos")
    .reset_index(name="count")
)
pos_df.head(15)


In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=pos_df.head(12), x="pos", y="count")
plt.title("POS-Verteilung im Gesamtkorpus")
plt.xlabel("POS")
plt.ylabel("Anzahl")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
sent_stats = (
    meta.groupby("corpus", dropna=False)["mean_sent_len"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

plt.figure(figsize=(10, 5))
sns.barplot(data=sent_stats, x="corpus", y="mean_sent_len")
plt.title("Mittlere Satzlänge nach Subkorpus")
plt.xlabel("Subkorpus")
plt.ylabel("Durchschnittliche Satzlänge")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 8a. Texte laden, falls sie schon durch LatinCy gelaufen sind

Hier laden wir die Texte in einen DataFrame, falls die Wortartenannotation und die Lemmatisierung bereits erfolgt sind.

In [ ]:
from pathlib import Path
from conllu import parse

conllu_dir = Path("outputs/conllu_letters")
files = sorted(conllu_dir.glob("*.conllu"))

allowed_upos = {"NOUN", "PROPN", "ADJ", "VERB"}

rows = []

for file in files:
    content = file.read_text(encoding="utf-8")
    
    try:
        sentences = parse(content)
    except Exception as e:
        print(f"Fehler bei {file.name}: {e}")
        continue

    lemmas_all = []
    lemmas_content = []

    for sent in sentences:
        for token in sent:
            if not isinstance(token["id"], int):
                continue

            lemma = token.get("lemma")
            upos = token.get("upostag") or token.get("upos")

            if lemma:
                lemmas_all.append(lemma)
                if upos in allowed_upos:
                    lemmas_content.append(lemma)

    if 'ad_atticum' in file.stem:
        corpus = 'ad_atticum'
    else:
        corpus = file.stem.split('.')[0]

    rows.append({
        "doc_id": file.stem,
        "corpus": corpus,
        "lemma_text": " ".join(lemmas_all),
        "content_lemma_text": " ".join(lemmas_content),
        "n_lemmas": len(lemmas_all),
        "n_content_lemmas": len(lemmas_content),
    })

df_loaded = pd.DataFrame(rows)

# diese Spalte dann für TF-IDF / Topic Modelling benutzen
df_loaded["clean_text"] = df_loaded["content_lemma_text"]

display(df_loaded.head())

In [ ]:
df_loaded.describe()

In [ ]:
df_loaded = df_loaded.merge(
    meta[["doc_id", "date_when"]],
    on="doc_id",
    how="left"
)

In [ ]:
df_loaded.head()

## 8b. Texte für Topic Modelling vorbereiten

Für Topic Modelling ist es meist sinnvoll, **nicht** die rohen Texte direkt zu nehmen, sondern eine leicht normalisierte Variante.

Hier bauen wir pro Brief einen Modelltext aus:
- **Lemmata**
- nur alphabetische Tokens
- ohne Interpunktion / Zahlen
- optional gefiltert auf inhaltstragende Wortarten

Das ist für lateinische Daten oft stabiler als rohe Wortformen.


In [ ]:
import re

LATIN_STOPWORDS = {
    "et", "in", "de", "ad", "non", "ut", "cum", "qui", "quae", "quod", "est", "esse",
    "sum", "ego", "tu", "nos", "vos", "hic", "ille", "is", "ea", "id", "autem", "enim",
    "sed", "si", "ne", "nec", "nam", "ita", "iam", "me", "te", "se", "mihi", "tibi",
    "sibi", "noster", "vester", "suus", "quid", "quoniam", "quoque", "tam", "tamen",
    "apud", "ab", "a", "ex", "e", "per", "pro", "post", "ante", "inter", "sine",
    "scribo", "littera"
}

def prepare_topic_text_from_string(text, min_len=2):
    if pd.isna(text):
        return ""

    lemmas = []
    for lemma in str(text).split():
        #lemma = lemma.lower().strip()
        lemma = re.sub(r"[^a-zA-Zāēīōūȳăĕĭŏŭ]+", "", lemma)

        if len(lemma) < min_len:
            continue
        if lemma in LATIN_STOPWORDS:
            continue

        lemmas.append(lemma)

    return " ".join(lemmas)

df_topics = df_loaded.copy()
df_topics["topic_text"] = df_topics["content_lemma_text"].apply(prepare_topic_text_from_string)

df_topics[["doc_id", "topic_text"]].head()

In [ ]:
# Leere oder fast leere Dokumente ausschliessen
mask_nonempty = df_topics["topic_text"].str.strip().str.len() > 0
df_topics = df_topics.loc[mask_nonempty].reset_index(drop=True)

len(df_topics), df_topics["topic_text"].str.split().str.len().describe()


## 9a. TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# -----------------------------
# 1. Daten vorbereiten
# -----------------------------
texts = df_topics["topic_text"].fillna("").tolist()
doc_ids = df_topics["doc_id"].tolist()

min_words = 5
valid_idx = [i for i, text in enumerate(texts) if len(text.split()) >= min_words]

texts_net = [texts[i] for i in valid_idx]
doc_ids_net = [doc_ids[i] for i in valid_idx]

print(f"Dokumente im Netzwerk: {len(doc_ids_net)}")

# -----------------------------
# 2. TF-IDF Matrix
# -----------------------------
vectorizer = TfidfVectorizer(
    token_pattern=r"(?u)\b\w+\b",
    min_df=3,          # Wort muss in mind. 3 Briefen vorkommen
    max_df=0.5,        # extrem häufige Wörter raus
    ngram_range=(1,2)
)

tfidf_matrix = vectorizer.fit_transform(texts_net)

print("Matrix shape:", tfidf_matrix.shape)

terms = vectorizer.get_feature_names_out()

# -----------------------------
# 3. Top Wörter pro Dokument
# -----------------------------
def top_tfidf_words(doc_index, n=10):
    row = tfidf_matrix[doc_index].toarray().flatten()
    top_idx = row.argsort()[::-1][:n]
    return [(terms[i], row[i]) for i in top_idx]

In [ ]:
doc_index = 0
print("Document:", doc_ids[doc_index])
print(top_tfidf_words(doc_index, 10))

In [ ]:
doc_index = 2
print("Document:", doc_ids[doc_index])
print(top_tfidf_words(doc_index, 10))

### Charakteristische Wörter pro Brief

In [ ]:
rows = []

for i, doc_id in enumerate(doc_ids_net):
    top_words = top_tfidf_words(i, 5)
    for word, score in top_words:
        rows.append({
            "doc_id": doc_id,
            "word": word,
            "tfidf": score
        })

tfidf_df = pd.DataFrame(rows)

display(tfidf_df.head(20))

### Wichtigste Wörter im gesamten Korpus

In [ ]:
mean_scores = np.asarray(tfidf_matrix.mean(axis=0)).flatten()

top_idx = mean_scores.argsort()[::-1][:20]

top_words = pd.DataFrame({
    "word": terms[top_idx],
    "score": mean_scores[top_idx]
})

display(top_words)

plt.figure(figsize=(8,6))
plt.barh(top_words["word"][::-1], top_words["score"][::-1])
plt.title("Most characteristic words in the corpus (TF-IDF)")
plt.xlabel("TF-IDF score")
plt.show()

### Die ähnlichsten Briefe basierend auf TF-IDF

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

sim_matrix = cosine_similarity(tfidf_matrix)

sim_df = pd.DataFrame(sim_matrix, index=doc_ids_net, columns=doc_ids_net)

display(sim_df.iloc[:10, :10])

In [ ]:
doc = doc_ids[800]

similar = sim_df.loc[doc].sort_values(ascending=False)[1:6]

display(similar)

In [ ]:
list(df_topics[df_topics['doc_id'] == 'ad_familiares_b6.0_l9_r88']['clean_text'])

In [ ]:
list(df_topics[df_topics['doc_id'] == 'ad_familiares_b13.0_l66_r345']['clean_text'])

In [ ]:
set(list(df_topics[df_topics['doc_id'] == 'ad_familiares_b6.0_l9_r88']['clean_text'])[0].split()).intersection(set(list(df_topics[df_topics['doc_id'] == 'ad_familiares_b13.0_l66_r345']['clean_text'])[0].split()))

### Briefe als Netzwerk

In [ ]:
import networkx as nx

In [ ]:
import plotly.graph_objects as go

print(f"Dokumente im Netzwerk: {len(doc_ids_net)}")

# -----------------------------
# 1. Netzwerk aufbauen
# -----------------------------
threshold = 0.20

G = nx.Graph()

for doc_id in doc_ids_net:
    G.add_node(doc_id)

n = len(doc_ids_net)
for i in range(n):
    for j in range(i + 1, n):
        sim = sim_matrix[i, j]
        if sim >= threshold:
            G.add_edge(doc_ids_net[i], doc_ids_net[j], weight=float(sim))

print(f"Knoten: {G.number_of_nodes()}")
print(f"Kanten: {G.number_of_edges()}")

# Optional: Isolates entfernen
G_plot = G.copy()
isolates = list(nx.isolates(G_plot))
G_plot.remove_nodes_from(isolates)

print(f"Ohne Isolates -> Knoten: {G_plot.number_of_nodes()}, Kanten: {G_plot.number_of_edges()}")

# -----------------------------
# 3. Layout berechnen
# -----------------------------
pos = nx.spring_layout(G_plot, seed=42, k=0.35)

# -----------------------------
# 4. Edge-Trace
# -----------------------------
edge_x = []
edge_y = []
edge_text = []

for u, v, data in G_plot.edges(data=True):
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])
    edge_text.append(f"{u} ↔ {v}<br>similarity={data['weight']:.3f}")

edge_trace = go.Scatter(
    x=edge_x,
    y=edge_y,
    line=dict(width=0.7, color="#888"),
    hoverinfo="none",
    mode="lines"
)

# -----------------------------
# 5. Node-Trace
# -----------------------------
node_x = []
node_y = []
node_text = []
node_size = []
node_color = []

degrees = dict(G_plot.degree())

for node in G_plot.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)

    deg = degrees[node]
    node_size.append(10 + deg * 3)
    node_color.append(deg)

    node_text.append(
        f"<b>{node}</b><br>"
        f"Degree: {deg}"
    )

corpus_map = dict(zip(df_topics["doc_id"], df_topics["corpus"]))

unique_corpora = sorted(set(corpus_map.get(node, "unknown") for node in G_plot.nodes()))
corpus_to_int = {c: i for i, c in enumerate(unique_corpora)}

node_color = [corpus_to_int.get(corpus_map.get(node, "unknown"), 0) for node in G_plot.nodes()]
node_text = [
    f"<b>{node}</b><br>Corpus: {corpus_map.get(node, 'unknown')}<br>Degree: {degrees[node]}"
    for node in G_plot.nodes()
]

node_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode="markers+text",
    text=["" for _ in G_plot.nodes()],   # leer; Labels nur bei Hover
    hoverinfo="text",
    hovertext=node_text,
    marker=dict(
    showscale=True,
    colorscale="Viridis",
    color=node_color,
    size=node_size,
    colorbar=dict(title="Corpus-ID"),
    line_width=1
)
)

# -----------------------------
# 6. Figure
# -----------------------------
fig = go.Figure(
    data=[edge_trace, node_trace],
    layout=go.Layout(
        title="Interaktives Netzwerk ähnlicher Briefe (TF-IDF + Cosine Similarity)",
        title_x=0.5,
        showlegend=False,
        hovermode="closest",
        margin=dict(b=20, l=20, r=20, t=50),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        height=800
    )
)

fig.show()

In [ ]:
from pyvis.network import Network

net = Network(height="800px", width="100%", notebook=True, cdn_resources="in_line")

for node in G_plot.nodes():
    net.add_node(
        node,
        label=node,
        title=f"{node}<br>Degree: {degrees[node]}",
        value=degrees[node]
    )

for u, v, data in G_plot.edges(data=True):
    net.add_edge(u, v, value=data["weight"], title=f"similarity={data['weight']:.3f}")

net.show("letter_network.html")

In [ ]:
list(df_topics[df_topics['doc_id'] == 'ad_familiares_b13.0_l38_r317']['clean_text'])

In [ ]:
list(df_topics[df_topics['doc_id'] == 'ad_familiares_b13.0_l46_r325']['clean_text'])

## 9b. BERTopic initialisieren

Für BERTopic verwenden wir:
- einen **SentenceTransformer** für Embeddings
- einen **CountVectorizer** für die c-TF-IDF-Komponente
- einige einfache Parameter, damit das Modell nicht zu fein granuliert

**Wichtiger methodischer Hinweis:**  
Für Latein gibt es nicht dieselbe bequeme Modelllandschaft wie für Englisch. Deshalb ist das hier ein **pragmatischer, explorativer Ansatz**. Für belastbare Forschung müsstest du verschiedene Einstellungen vergleichen und die Topics historisch-philologisch validieren.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

try:
    from sentence_transformers import SentenceTransformer
    embedding_model = SentenceTransformer("LaBSE")
    print("Embedding-Modell geladen.")
except Exception as e:
    embedding_model = None
    print("Kein SentenceTransformer geladen. BERTopic läuft dann nur mit Vektorraum-/c-TF-IDF-Komponenten.")
    print(type(e).__name__, e)

In [ ]:
from bertopic import BERTopic
from hdbscan import HDBSCAN

In [ ]:
from umap import UMAP

In [ ]:
umap_model = UMAP(
    n_neighbors=15,      # eher 15–30 testen
    n_components=5,      # für Clustering oft stabiler als 2/3
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=8,   # eher 12–20 testen
    min_samples=3,         # konservativer als None
    metric="euclidean",    # auf UMAP-Reduktion üblich
    cluster_selection_method="eom",
    prediction_data=True
)

vectorizer_model = CountVectorizer(
    lowercase=False,                 # falls du schon normalisierte Lemmata hast
    token_pattern=r"(?u)\b\w+\b",
    ngram_range=(1, 2),              # bei Cicero oft hilfreich
    min_df=3,                        # bei sehr kleinem Korpus evtl. 2
    max_df=0.6
)

In [ ]:
topic_model = BERTopic(
    embedding_model=embedding_model,
    hdbscan_model=hdbscan_model,
    umap_model=umap_model,
    vectorizer_model=vectorizer_model,
    #language="multilingual",
    calculate_probabilities=True,
    verbose=True,
)

topics, probs = topic_model.fit_transform(df_topics["topic_text"])
df_topics["topic"] = topics
df_topics["topic_prob"] = [float(np.max(p)) if p is not None else np.nan for p in probs]
df_topics[["doc_id", "topic", "topic_prob", "topic_text"]].head()


## 10. Erste Sicht auf die Topics


In [ ]:
topic_info = topic_model.get_topic_info()
topic_info.head(20)


In [ ]:
# Lesbare Top-Wörter pro Topic
for topic_id in topic_info["Topic"].head(12):
    if topic_id == -1:
        continue
    print(f"\nTOPIC {topic_id}")
    print(topic_model.get_topic(topic_id)[:12])


## 11. Topic-Grössen visualisieren


In [ ]:
topic_sizes = topic_info[topic_info["Topic"] != -1].copy()

plt.figure(figsize=(10, 6))
sns.barplot(data=topic_sizes, x="Topic", y="Count")
plt.title("Grösste Topics im Korpus")
plt.xlabel("Topic")
plt.ylabel("Anzahl Briefe")
plt.tight_layout()
plt.show()


## 12. Interaktive BERTopic-Visualisierungen

Die folgenden Visualisierungen sind besonders nützlich:
- **Topic map**: räumliche Nähe zwischen Topics
- **Barchart**: wichtigste Wörter pro Topic
- **Hierarchy**: Themen-Hierarchie
- **Heatmap**: Ähnlichkeit zwischen Topics

In Jupyter werden sie interaktiv angezeigt. Zusätzlich speichern wir sie als HTML.


In [ ]:
fig_topics = topic_model.visualize_topics()
fig_topics


In [ ]:
fig_barchart = topic_model.visualize_barchart(top_n_topics=12)
fig_barchart


In [ ]:
fig_heatmap = topic_model.visualize_heatmap()
fig_heatmap


In [ ]:
fig_hierarchy = topic_model.visualize_hierarchy()
fig_hierarchy


In [ ]:
fig_topics.write_html(str(topic_dir / "bertopic_topics_map.html"))
fig_barchart.write_html(str(topic_dir / "bertopic_barchart.html"))
fig_heatmap.write_html(str(topic_dir / "bertopic_heatmap.html"))
fig_hierarchy.write_html(str(topic_dir / "bertopic_hierarchy.html"))

sorted(topic_dir.glob("*.html"))


## 13. Topics pro Subkorpus

Jetzt prüfen wir, ob einzelne Subkorpora andere Topic-Schwerpunkte haben.


In [ ]:
topic_by_corpus = (
    df_topics.groupby(["corpus", "topic"])
    .size()
    .reset_index(name="n")
)

topic_pivot = topic_by_corpus.pivot(index="corpus", columns="topic", values="n").fillna(0)
topic_pivot


In [ ]:
topic_share = topic_pivot.div(topic_pivot.sum(axis=1), axis=0)

plt.figure(figsize=(12, 6))
sns.heatmap(topic_share, cmap="Blues")
plt.title("Topic-Anteile nach Subkorpus")
plt.xlabel("Topic")
plt.ylabel("Subkorpus")
plt.tight_layout()
plt.show()


## 14. Topics über die Zeit

Falls dein Datensatz eine brauchbare Datumsangabe enthält, kannst du Topics auch zeitlich untersuchen.

Da antike Datierungen oft unregelmässig sind, gehen wir bewusst defensiv vor:
- wir extrahieren, wenn möglich, ein Jahr aus `date_when`
- dann zählen wir Topics pro Jahr


In [ ]:
def extract_year(value):
    if pd.isna(value):
        return np.nan
    value = str(value)

    # einfache Heuristik: erste 1-4 Ziffern greifen
    m = re.search(r"(-?\d{1,4})", value)
    if m:
        try:
            return int(m.group(1))
        except Exception:
            return np.nan
    return np.nan

if "date_when" in df_topics.columns:
    df_topics["year_simple"] = df_topics["date_when"].apply(extract_year)
else:
    df_topics["year_simple"] = np.nan

df_topics[["date_when", "year_simple"]].head(10)


In [ ]:
time_df = df_topics.dropna(subset=["year_simple"]).copy()
time_df["year_simple"] = time_df["year_simple"].astype(int)

topic_year = (
    time_df.groupby(["year_simple", "topic"])
    .size()
    .reset_index(name="n")
)

topic_year.head()


In [ ]:
# Nur die häufigsten Topics darstellen
top_topics = topic_info[topic_info["Topic"] != -1]["Topic"].tolist()
plot_df = topic_year[topic_year["topic"].isin(top_topics)]

plt.figure(figsize=(12, 6))
sns.lineplot(data=plot_df, x="year_simple", y="n", hue="topic", marker="o")
plt.title("Ausgewählte Topics über die Zeit")
plt.xlabel("Jahr")
plt.ylabel("Anzahl Briefe")
plt.tight_layout()
plt.show()


## 15. Repräsentative Briefe pro Topic finden

Für die Interpretation ist es hilfreich, pro Topic ein paar Beispielbriefe anzuschauen.


In [ ]:
rep_docs = []
for topic_id in sorted(df_topics["topic"].unique()):
    if topic_id == -1:
        continue
    sub = df_topics[df_topics["topic"] == topic_id].copy()
    sub = sub.sort_values("topic_prob", ascending=False).head(3)
    rep_docs.append(sub[["doc_id", "corpus", "topic", "topic_prob", "topic_text"]])

rep_docs_df = pd.concat(rep_docs, ignore_index=True) if rep_docs else pd.DataFrame()
rep_docs_df.head(20)


## 16. Ergebnisse speichern

Wir sichern:
- Topic-Zuweisungen pro Brief
- Topic-Informationen
- repräsentative Briefe


In [ ]:
df_topics.to_csv(topic_dir / "letters_with_topics.csv", index=False)
topic_info.to_csv(topic_dir / "topic_info.csv", index=False)
rep_docs_df.to_csv(topic_dir / "representative_docs_per_topic.csv", index=False)

sorted(topic_dir.glob("*.csv"))


## 17. Methodische Reflexion

Ein paar Punkte, die du bei der Interpretation im Hinterkopf behalten solltest:

1. **Topics sind keine objektiven Wahrheiten**, sondern modellierte Wortbündel.
2. Bei **lateinischen Texten** hängt sehr viel an der Qualität von Lemmatisierung und POS-Tagging.
3. BERTopic ist stark, aber nicht magisch:
   - andere Parameter erzeugen andere Topics
   - auch die Vorverarbeitung beeinflusst das Ergebnis massiv
4. Gerade für den Kurs ist die Kombination aus
   - **philologischer Nahlektüre**
   - **linguistischer Vorverarbeitung**
   - **explorativer Modellierung**
   besonders fruchtbar.

Mit Blick auf White liesse sich nun z.B. fragen:
- Dominieren politische Topics?
- Unterscheiden sich Topics je nach Subkorpus?
- Entstehen Topic-Cluster, die editorische oder historische Episoden spiegeln?


## 18. Topic Modelling mit gensim

In [ ]:
import pandas as pd
from gensim import corpora, models
from gensim.models import CoherenceModel
from pprint import pprint
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

# =========================
# 1. Texte vorbereiten
# =========================
text_col = "topic_text"   # ggf. anpassen
id_col = "doc_id"         # ggf. anpassen

# nur brauchbare Texte behalten
df_lda = df_topics[[id_col, text_col]].dropna().copy()
df_lda = df_lda[df_lda[text_col].str.strip() != ""]

# tokenisieren
texts = [text.split() for text in df_lda[text_col]]

# sehr kurze Dokumente entfernen
min_tokens = 5
valid_mask = [len(toks) >= min_tokens for toks in texts]
texts = [toks for toks, keep in zip(texts, valid_mask) if keep]
df_lda = df_lda.loc[df_lda.index[valid_mask]].reset_index(drop=True)

print(f"Dokumente für LDA: {len(texts)}")

# =========================
# 2. Dictionary + Corpus
# =========================
dictionary = corpora.Dictionary(texts)

# Extremwerte filtern
# no_below: mindestens in X Dokumenten
# no_above: in höchstens Y Anteil der Dokumente
dictionary.filter_extremes(no_below=5, no_above=0.5, keep_n=5000)

corpus = [dictionary.doc2bow(text) for text in texts]

print(f"Vokabulargrösse: {len(dictionary)}")

# =========================
# 3. LDA trainieren
# =========================
num_topics = 20  # ggf. variieren

lda_model = models.LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=num_topics,
    random_state=42,
    chunksize=100,
    passes=20,
    iterations=400,
    alpha="auto",
    eta="auto",
    per_word_topics=True
)

# =========================
# 4. Topics ausgeben
# =========================
print("\nTopics:\n")
for i, topic in lda_model.print_topics(num_topics=num_topics, num_words=10):
    print(f"Topic {i}: {topic}")

# =========================
# 5. Kohärenz berechnen
# =========================
coherence_model = CoherenceModel(
    model=lda_model,
    texts=texts,
    dictionary=dictionary,
    coherence="c_v"
)
coherence = coherence_model.get_coherence()
print(f"\nCoherence (c_v): {coherence:.4f}")

# =========================
# 6. Dominantes Topic pro Dokument
# =========================
def get_dominant_topic(bow):
    topic_probs = lda_model.get_document_topics(bow, minimum_probability=0.0)
    dominant_topic, dominant_prob = max(topic_probs, key=lambda x: x[1])
    return dominant_topic, dominant_prob

dominant_topic = [get_dominant_topic(bow) for bow in corpus]
print(dominant_topic)
df_lda["dominant_topic"] = [t[0] for t in dominant]
df_lda["topic_prob"] = [t[1] for t in dominant]

display(df_lda.head())

# =========================
# 7. Interaktive Visualisierung
# =========================
vis = gensimvis.prepare(lda_model, corpus, dictionary)
pyLDAvis.display(vis)

## 18. Mini-Übungen

1. Verändere die Funktion `prepare_topic_text()`:
   - einmal nur `NOUN` + `PROPN`
   - einmal mit allen alphabetischen Tokens  
   Vergleiche die Topics.

2. Setze `min_topic_size` höher oder tiefer.  
   Wie verändert das die Zahl und Lesbarkeit der Topics?

3. Suche zu **einem** Topic drei Briefe heraus und prüfe per Nahlektüre:
   - Ist das Topic historisch sinnvoll?
   - Ist es zu allgemein?
   - Sind mehrere Topics eigentlich dasselbe?

4. Prüfe, ob ein bestimmtes Subkorpus (z.B. `ad_atticum`) stärker auf einzelne Topics konzentriert ist als andere.


In [ ]:
# -----------------------------
# 1. Document-topic matrix
# -----------------------------
doc_topic_matrix = []
for bow in corpus:
    topic_probs = lda_model.get_document_topics(bow, minimum_probability=0.0)
    doc_topic_matrix.append([prob for _, prob in topic_probs])

doc_topic_matrix = np.array(doc_topic_matrix)

topic_cols = [f"topic_{i}" for i in range(lda_model.num_topics)]
doc_topic_df = pd.DataFrame(doc_topic_matrix, columns=topic_cols)
doc_topic_df.insert(0, "doc_id", df_lda["doc_id"].values)

display(doc_topic_df.head())


In [ ]:
# -----------------------------
# 2. Topic sizes
# -----------------------------
topic_sizes = doc_topic_matrix.sum(axis=0)
topic_size_df = pd.DataFrame({
    "topic": range(lda_model.num_topics),
    "size": topic_sizes
}).sort_values("size", ascending=False)

plt.figure(figsize=(10, 5))
plt.bar(topic_size_df["topic"].astype(str), topic_size_df["size"])
plt.xlabel("Topic")
plt.ylabel("Total topic weight across corpus")
plt.title("Topic prevalence in the corpus")
plt.xticks(rotation=0)
plt.show()

display(topic_size_df)

In [ ]:
# -----------------------------
# 3. Top words per topic
# -----------------------------
top_n = 10
topic_word_rows = []

for topic_id in range(lda_model.num_topics):
    words = lda_model.show_topic(topic_id, topn=top_n)
    for rank, (word, weight) in enumerate(words, start=1):
        topic_word_rows.append({
            "topic": topic_id,
            "rank": rank,
            "word": word,
            "weight": weight
        })

topic_words_df = pd.DataFrame(topic_word_rows)
display(topic_words_df.head(30))

# small multiples: top words per topic
n_topics = lda_model.num_topics
ncols = 2
nrows = int(np.ceil(n_topics / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
axes = np.array(axes).reshape(-1)

for topic_id in range(n_topics):
    ax = axes[topic_id]
    sub = topic_words_df[topic_words_df["topic"] == topic_id].sort_values("weight", ascending=True)
    ax.barh(sub["word"], sub["weight"])
    ax.set_title(f"Topic {topic_id}")
    ax.set_xlabel("Weight")

for j in range(n_topics, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------
# 5. Document-topic heatmap
# -----------------------------
# For readability: show only first N docs
max_docs = min(50, len(doc_topic_df))
heatmap_data = doc_topic_matrix[:max_docs]

plt.figure(figsize=(12, 8))
plt.imshow(heatmap_data, aspect="auto")
plt.colorbar(label="Topic probability")
plt.xlabel("Topic")
plt.ylabel("Document")
plt.title(f"Document-topic heatmap (first {max_docs} documents)")
plt.xticks(range(lda_model.num_topics), range(lda_model.num_topics))
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
# -----------------------------
# 6. 2D document map (BERTopic-like document scatter)
# -----------------------------
pca = PCA(n_components=2, random_state=42)
doc_2d = pca.fit_transform(doc_topic_matrix)

plt.figure(figsize=(10, 7))
scatter = plt.scatter(
    doc_2d[:, 0],
    doc_2d[:, 1],
    c=[dominant_topic,
    s=30 + dominant_prob * 120,
    alpha=0.75
)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Documents in topic space (2D projection)")
plt.colorbar(scatter, label="Dominant topic")
plt.show()

In [ ]:
# -----------------------------
# 7. Topic similarity heatmap
# -----------------------------
# topic-word matrix from lda_model.get_topics()
topic_word_matrix = lda_model.get_topics()
topic_similarity = cosine_similarity(topic_word_matrix)

plt.figure(figsize=(8, 6))
plt.imshow(topic_similarity, aspect="auto")
plt.colorbar(label="Cosine similarity")
plt.xticks(range(n_topics), range(n_topics))
plt.yticks(range(n_topics), range(n_topics))
plt.xlabel("Topic")
plt.ylabel("Topic")
plt.title("Topic similarity matrix")
plt.show()